In [ ]:
from snowflake.snowpark.functions import col
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER"
    ).filter(col("C_NATIONKEY") == '15'
    ).select(col("C_CUSTKEY"), col("C_NAME"))
df

In [ ]:
df.schema
df.schema.names

In [ ]:
df.write.mode("overwrite").save_as_table("exercise_db.public.customer")
session.table("exercise_db.public.customer").count()

In [ ]:
df.write.mode("append").save_as_table("exercise_db.public.customer")
session.table("exercise_db.public.customer").count()

In [ ]:
df.write.mode("ignore").save_as_table("exercise_db.public.customer")
session.table("exercise_db.public.customer").count()

In [ ]:
df.write.mode("errorifexists").save_as_table("exercise_db.public.customer")
session.table("exercise_db.public.customer").count()

In [ ]:
df.write.mode("errorifexists").save_as_table("exercise_db.public.customer_t", table_type="temp")
session.table("exercise_db.public.customer_t").count()

In [ ]:
view = df.create_or_replace_view("exercise_db.public.customer_vw")
session.table("exercise_db.public.customer_vw").count()

In [ ]:
df.write.copy_into_location(
    "@exercise_db.public.stage1/customer.parquet",
    file_format_type="parquet",
    header=True, overwrite=True, single=True)

In [ ]:
list @exercise_db.public.stage1

In [ ]:
df2 = session.read.parquet("@exercise_db.public.stage1/customer.parquet")
df2

In [ ]:
# recreate table
session.sql("drop table if exists exercise_db.public.customer_f").collect()

# COPY INTO table (from dataframe)
df2.copy_into_table("exercise_db.public.customer_f", force=True)

# load table data
df = session.table("exercise_db.public.customer_f")
df

In [ ]:
# create and populate a temp table (for cache_result())
session.sql("create or replace temp table cache_t(num int)").collect()
session.sql("insert into cache_t values (1), (2)").collect()
df = session.table("cache_t")
df

In [ ]:
print("Database:", session.get_current_database())
print("Schema:", session.get_current_schema())


In [ ]:
# cache df into a temp table
df_cached = df.cache_result()
df_cached.is_cached
df_cached

In [ ]:
# the cached result does not change!
session.sql("insert into cache_t values (3)").collect()
df
df_cached

In [ ]:
# clear cached result
df_cached.drop_table()
df_cached